# Day 2 Lab – Semantic Document Search

**Objective:**
Build a system that (1) takes a set of documents, (2) embeds them with a Hugging Face / sentence-transformers model, (3) builds a FAISS index, (4) answers natural-language queries with top-k similar documents.

**Prerequisites:** Day 2 Modules 01–03. Duration: ~30–45 min.

**Scenario:** Build a prototype semantic search engine for an internal support knowledge base. The company's IT helpdesk has ~20 articles. Employees type a question; the system returns the most relevant articles without requiring exact keyword matches.

## Pipeline overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                 SEMANTIC DOCUMENT SEARCH PIPELINE                           │
│                                                                             │
│  ┌─────────────┐   ┌──────────────┐   ┌───────────────┐                   │
│  │  Block 1     │   │  Block 2      │   │  Block 3       │                  │
│  │  LOAD DOCS   │──►│  EMBED DOCS   │──►│  BUILD INDEX   │                  │
│  │              │   │              │   │  (FAISS)       │                   │
│  │  20 articles │   │  model.encode│   │  IndexFlatL2   │                   │
│  └─────────────┘   └──────────────┘   └───────┬───────┘                   │
│                                                │                            │
│                                       index stored in memory                │
│                                                │                            │
│  ┌─────────────┐   ┌──────────────┐   ┌───────▼───────┐                   │
│  │  Block 6     │   │  Block 5      │   │  Block 4       │                  │
│  │  DISPLAY     │◄──│  MAP TO DOCS  │◄──│  EMBED & SEARCH│                  │
│  │  RESULTS     │   │  indices→text │   │  query→vector  │                  │
│  │              │   │              │   │  index.search() │                  │
│  └─────────────┘   └──────────────┘   └───────────────┘                   │
│                                                                             │
│  Each code section below is labeled with its Block number.                  │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Setup

In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

with open(Path.cwd() / "config.json") as f:
    CONFIG = json.load(f)

print("Config loaded. Embedding model:", CONFIG["embedding_model"])

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print(f"faiss version: {faiss.__version__ if hasattr(faiss, '__version__') else 'N/A'}")
print(f"numpy: {np.__version__}")

## Block 1: Load and preprocess documents

The knowledge base contains IT support articles. In production, these would come from a database, CMS, or file system. Here we use inline strings to keep the lab self-contained.

In [ ]:
# Block 1: Load documents — IT helpdesk knowledge base
documents = [
    "Password Reset: To reset your password, navigate to Settings > Security > Change Password. Enter your current password, then your new password twice. You will receive a confirmation email. If you forgot your current password, click 'Forgot Password' on the login page to receive a reset link.",
    "VPN Setup: Install the company VPN client from the IT portal (Software > VPN Client). After installation, open the client and enter the server address: vpn.company.internal. Authenticate with your corporate username and MFA token. The VPN is required for accessing internal resources from outside the office.",
    "Hardware Request: To request new hardware (laptop, monitor, keyboard, mouse), submit a ticket through the IT portal under Hardware Requests. Include your department, manager name, and justification. Standard hardware is pre-approved and ships within 5 business days. Non-standard requests require VP approval.",
    "Email Configuration: The company uses Microsoft 365 for email. On mobile devices, install the Microsoft Outlook app and sign in with your corporate email address. The app will auto-configure server settings. For desktop, Outlook is pre-installed on company laptops. Use your corporate credentials to sign in.",
    "Expense Reporting: All business expenses must be submitted through the expense management portal within 30 days of the transaction. Attach scanned receipts for all expenses over $25. Travel expenses require pre-approval from your manager. Reimbursements are processed in the next payroll cycle.",
    "Internal Wiki Access: The company wiki is available at wiki.company.internal. Access requires connection to the corporate network (office LAN or VPN). Search for topics using the search bar. To create or edit pages, you need Editor permissions — request access through your manager.",
    "Software Installation: Standard software (Office, Slack, Zoom, VS Code) is available for self-install from the IT portal. Non-standard software requires a ticket with business justification and manager approval. Installation of unauthorized software is a policy violation.",
    "Multi-Factor Authentication: MFA is mandatory for all corporate accounts. Set up MFA using the Microsoft Authenticator app: go to Security Settings > MFA > Add Method > Authenticator App, then scan the QR code. You will need MFA for VPN, email on new devices, and sensitive applications.",
    "Shared Drive Access: The company shared drive is mounted at S:\\ on office computers. To access remotely, connect to VPN first, then map the network drive: File Explorer > Map Network Drive > drive letter S: > path \\\\fileserver\\shared. Contact IT if you cannot see specific department folders.",
    "Printer Setup: Company printers are auto-discovered on the office network. Go to Settings > Printers > Add Printer to find available printers. For printing from home, use the web print portal at print.company.internal (VPN required). If a printer is jammed or out of toner, submit a facilities ticket.",
    "Conference Room Booking: Book conference rooms through the Outlook calendar. Create a new meeting and add the room as a location. Rooms with 'VC' in the name have video conferencing equipment. For rooms seating 20+, book at least 48 hours in advance. Cancel bookings you no longer need.",
    "Guest WiFi: The guest WiFi network is 'Company-Guest'. It provides internet-only access with no connection to internal resources. Guests must accept the terms of use on the captive portal. For vendor access to internal systems, submit a Temporary Access Request through the IT portal.",
    "Data Backup: Company data on laptops is automatically backed up to the cloud every 24 hours when connected to the internet. The backup software runs in the background. To restore files, open the backup client from the system tray and select Restore. For server data recovery, contact the infrastructure team.",
    "Security Incident Reporting: If you suspect a security incident (phishing email, unauthorized access, lost device, suspicious activity), report it immediately to security@company.internal or call the Security Operations Center at ext. 5555. Do not forward suspected phishing emails — screenshot and report.",
    "New Employee Onboarding IT: New employees receive their laptop and credentials on Day 1. Your manager will provide a setup guide. Complete the IT Security Awareness Training within 7 days. Set up MFA, VPN, and email in that order. The IT portal has a 'New Employee Checklist' with step-by-step instructions.",
    "Remote Work Policy: Remote employees must use the company VPN for all work. Ensure your home network uses WPA3 or WPA2 encryption. Do not use public WiFi for work without VPN. Company data must not be stored on personal devices. Use the company laptop for all work activities.",
    "Software License Management: All software licenses are managed centrally by IT. Do not purchase software licenses independently. For license requests, submit a ticket with the software name, version, and number of users. Unused licenses are reclaimed after 90 days of inactivity.",
    "Account Lockout: After 5 failed login attempts, your account is automatically locked for 30 minutes. If you need immediate access, contact the IT helpdesk at ext. 4444 or helpdesk@company.internal. Repeated lockouts may trigger a security review of your account.",
    "Mobile Device Management: Company mobile devices are enrolled in the MDM system. Do not unenroll your device. MDM allows remote lock and wipe in case of loss or theft. Report lost or stolen devices immediately to the IT helpdesk. Personal apps on company devices must comply with the acceptable use policy.",
    "System Maintenance Windows: Planned system maintenance occurs every Sunday 2:00-6:00 AM. During this window, internal applications and VPN may be unavailable. Emergency maintenance is communicated via the #it-announcements Slack channel. Subscribe to the channel for real-time updates.",
]

print(f"Knowledge base: {len(documents)} articles")
print(f"Average article length: {sum(len(d) for d in documents) / len(documents):.0f} chars")

# Preview
for i, doc in enumerate(documents[:3]):
    title = doc.split(":")[0]
    print(f"  [{i:2d}] {title}")
print(f"  ... and {len(documents) - 3} more")

## Block 2: Embed documents

Convert all documents to vectors using the sentence-transformers model from `config.json`.

In [ ]:
# Block 2: Embed all documents
model = SentenceTransformer(CONFIG["embedding_model"])

start = time.perf_counter()
doc_embeddings = model.encode(documents, show_progress_bar=True, convert_to_numpy=True)
embed_time = time.perf_counter() - start

doc_embeddings = doc_embeddings.astype("float32")

print(f"\nEmbedding complete:")
print(f"  Model:      {CONFIG['embedding_model']}")
print(f"  Documents:  {doc_embeddings.shape[0]}")
print(f"  Dimensions: {doc_embeddings.shape[1]}")
print(f"  Time:       {embed_time:.2f}s ({embed_time/len(documents)*1000:.1f}ms per doc)")
print(f"  Memory:     {doc_embeddings.nbytes / 1024:.1f} KB")

## Block 3: Build FAISS index

Create a FAISS index and add all document vectors. `IndexFlatL2` performs exact search — appropriate for datasets up to ~100K documents.

In [ ]:
# Block 3: Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"FAISS index built:")
print(f"  Type:    IndexFlatL2 (exact search)")
print(f"  Vectors: {index.ntotal}")
print(f"  Dims:    {dimension}")

## Blocks 4–6: Query, search, and display results

Embed the query, search the FAISS index for nearest neighbors, map indices back to document text, and display.

In [ ]:
# Blocks 4-6: Search function
def search(query: str, k: int = 3) -> list[dict]:
    """Semantic search: embed query → FAISS search → return ranked docs."""
    # Block 4: Embed query
    q_vec = model.encode([query]).astype("float32")
    
    # Block 5: Search index
    distances, indices = index.search(q_vec, k)
    
    # Block 6: Map to documents
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            "rank": rank + 1,
            "index": int(idx),
            "distance": float(dist),
            "title": documents[idx].split(":")[0],
            "text": documents[idx],
        })
    return results

In [ ]:
# First query
query = "I forgot my password and can't log in"
results = search(query, k=3)

print(f"Query: '{query}'\n")
for r in results:
    print(f"  {r['rank']}. [{r['title']}]  (dist={r['distance']:.4f})")
    print(f"     {r['text'][:100]}...")
    print()

In [ ]:
# More queries — testing different topics
test_queries = [
    "How do I connect to the office network from home?",
    "I need a new laptop for work",
    "Where can I find the company wiki?",
    "What should I do if I get a phishing email?",
    "How to set up two-factor authentication?",
    "Can my guest use the WiFi?",
]

for q in test_queries:
    results = search(q, k=2)
    print(f"Q: {q}")
    for r in results:
        print(f"   {r['rank']}. [{r['title']}]  dist={r['distance']:.4f}")
    print()

## Evaluating search quality

Measure how often the correct document appears in the top-k results.

In [ ]:
# Ground truth: query → expected document index
ground_truth = [
    ("How do I reset my password?", 0),
    ("I need VPN access", 1),
    ("How to order new hardware", 2),
    ("Set up email on my phone", 3),
    ("How to submit expenses", 4),
    ("Where is the company wiki?", 5),
    ("Install software on my laptop", 6),
    ("How to enable MFA", 7),
    ("Access the shared drive remotely", 8),
    ("Printer is not working", 9),
    ("Book a conference room", 10),
    ("WiFi for visitors", 11),
    ("How does data backup work?", 12),
    ("Report a security incident", 13),
    ("I'm a new employee, what IT setup do I need?", 14),
    ("Working from home policy", 15),
]

# Evaluate recall@k for k=1, 3, 5
for k in [1, 3, 5]:
    hits = 0
    for query, expected_idx in ground_truth:
        results = search(query, k=k)
        found = any(r["index"] == expected_idx for r in results)
        hits += int(found)
    recall = hits / len(ground_truth)
    print(f"  Recall@{k}: {hits}/{len(ground_truth)} = {recall:.0%}")

In [ ]:
# Detailed breakdown — show misses
print("Detailed results (Recall@3):\n")
for query, expected_idx in ground_truth:
    results = search(query, k=3)
    found_indices = [r["index"] for r in results]
    hit = expected_idx in found_indices
    marker = "HIT " if hit else "MISS"
    expected_title = documents[expected_idx].split(":")[0]
    print(f"  [{marker}] '{query[:45]:45s}' expected=[{expected_title}] found={found_indices}")

In [ ]:
# Search latency benchmark
latencies = []
for q, _ in ground_truth:
    start = time.perf_counter()
    _ = search(q, k=3)
    latencies.append(time.perf_counter() - start)

avg_ms = sum(latencies) / len(latencies) * 1000
min_ms = min(latencies) * 1000
max_ms = max(latencies) * 1000

print(f"Search latency ({len(documents)} docs, {dimension}d):")
print(f"  Average: {avg_ms:.1f}ms")
print(f"  Min:     {min_ms:.1f}ms")
print(f"  Max:     {max_ms:.1f}ms")
print(f"  QPS:     {1000/avg_ms:.0f} queries/sec")

In [ ]:
# Analyze distance distributions for hits vs misses
hit_dists = []
miss_dists = []

for query, expected_idx in ground_truth:
    results = search(query, k=3)
    for r in results:
        if r["index"] == expected_idx:
            hit_dists.append(r["distance"])
        else:
            miss_dists.append(r["distance"])

print("Distance distribution:")
print(f"  Hits  — mean={np.mean(hit_dists):.4f}, min={np.min(hit_dists):.4f}, max={np.max(hit_dists):.4f}")
print(f"  Other — mean={np.mean(miss_dists):.4f}, min={np.min(miss_dists):.4f}, max={np.max(miss_dists):.4f}")
print(f"\nLower distance = closer match. Good separation between hits and non-hits indicates quality.")

## Saving and loading the index

In production, you embed documents once and save the index to disk. On restart, load the index instead of re-embedding.

In [ ]:
import tempfile

# Save FAISS index
index_path = Path(tempfile.mkdtemp()) / "helpdesk.faiss"
faiss.write_index(index, str(index_path))
print(f"Index saved: {index_path} ({index_path.stat().st_size / 1024:.1f} KB)")

# Save document mapping (needed to map indices back to text)
import pickle
docs_path = index_path.with_suffix(".pkl")
with open(docs_path, "wb") as f:
    pickle.dump(documents, f)
print(f"Docs saved:  {docs_path} ({docs_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Load and verify
loaded_index = faiss.read_index(str(index_path))
with open(docs_path, "rb") as f:
    loaded_docs = pickle.load(f)

# Verify search produces same results
q_vec = model.encode(["VPN setup"]).astype("float32")
D_orig, I_orig = index.search(q_vec, k=3)
D_loaded, I_loaded = loaded_index.search(q_vec, k=3)

assert (I_orig == I_loaded).all()
print(f"Loaded index: {loaded_index.ntotal} vectors — verified identical results")

# Cleanup
import shutil
shutil.rmtree(index_path.parent)

## Adding new documents to the index

When new articles are published, embed and add them to the existing index without rebuilding from scratch.

In [ ]:
# Add new documents
new_articles = [
    "Building Access Cards: Lost or damaged building access cards can be replaced at the Security Desk in the lobby. Bring a photo ID. Temporary cards are issued same-day; permanent replacements take 3 business days. Report lost cards immediately — lost cards are deactivated within 1 hour.",
    "IT Training Resources: The company provides free access to LinkedIn Learning, Pluralsight, and internal training modules. Access through the Learning Portal at learn.company.internal. Complete at least 2 courses per quarter for your professional development goals.",
]

# Embed new documents
new_embeddings = model.encode(new_articles, convert_to_numpy=True).astype("float32")

# Add to existing structures
documents.extend(new_articles)
index.add(new_embeddings)

print(f"Added {len(new_articles)} new articles")
print(f"Total index size: {index.ntotal}")

# Test: search for new content
results = search("I lost my badge", k=2)
print(f"\nQuery: 'I lost my badge'")
for r in results:
    print(f"  {r['rank']}. [{r['title']}] dist={r['distance']:.4f}")

## Enhanced search display

Format results for readability — display article title, relevance score (converted from distance), and a text snippet.

In [ ]:
def search_and_display(query: str, k: int = 3) -> None:
    """Search and print formatted results."""
    results = search(query, k=k)
    
    # Convert L2 distance to a 0-1 relevance score (approximate)
    max_dist = max(r["distance"] for r in results) if results else 1.0
    
    print(f"Query: '{query}'")
    print(f"{'─' * 80}")
    for r in results:
        # Simple relevance: 1 / (1 + distance)
        relevance = 1 / (1 + r["distance"])
        snippet = r["text"][:150].replace("\n", " ")
        print(f"  {r['rank']}. {r['title']}")
        print(f"     Relevance: {relevance:.3f}  |  Distance: {r['distance']:.4f}")
        print(f"     {snippet}...")
        print()


search_and_display("How do I set up two-factor authentication?")

In [ ]:
# Interactive-style queries
queries_interactive = [
    "my account is locked",
    "can I install Photoshop on my work laptop?",
    "when is system maintenance?",
    "how to back up my files",
    "I lost my company phone",
]

for q in queries_interactive:
    search_and_display(q, k=2)
    print()

## Alternative: Chroma implementation

The same search pipeline using Chroma instead of FAISS. Chroma handles embedding and indexing internally and supports metadata filtering.

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="helpdesk_kb",
    metadata={"hnsw:space": "cosine"},
)

# Extract titles as metadata
doc_titles = [doc.split(":")[0] for doc in documents]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"title": t} for t in doc_titles],
)

print(f"Chroma collection: {collection.count()} documents")

In [ ]:
# Search with Chroma
def search_chroma(query: str, k: int = 3) -> None:
    results = collection.query(query_texts=[query], n_results=k)
    
    print(f"Query: '{query}'")
    print(f"{'─' * 80}")
    for doc, dist, meta, doc_id in zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0],
        results["ids"][0],
    ):
        print(f"  [{doc_id}] {meta['title']}")
        print(f"  Distance: {dist:.4f}  |  {doc[:120]}...")
        print()


search_chroma("How do I reset my password?")

In [ ]:
# Chroma with metadata filter
# Add categories for a subset
categories = [
    "account", "network", "hardware", "email", "finance",
    "knowledge", "software", "account", "storage", "hardware",
    "facilities", "network", "backup", "security", "onboarding",
    "policy", "software", "account", "device", "infrastructure",
]

# Update collection with category metadata
collection_with_cats = chroma_client.create_collection(
    name="helpdesk_kb_categories",
    metadata={"hnsw:space": "cosine"},
)
collection_with_cats.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"title": doc_titles[i], "category": categories[i]} for i in range(len(documents))],
)

# Filter by category
results_filtered = collection_with_cats.query(
    query_texts=["login problem"],
    n_results=3,
    where={"category": "account"},
)

print("Query: 'login problem' (category=account)\n")
for doc, dist, meta in zip(
    results_filtered["documents"][0],
    results_filtered["distances"][0],
    results_filtered["metadatas"][0],
):
    print(f"  [{meta['category']}] {meta['title']}")
    print(f"  dist={dist:.4f}  {doc[:100]}...\n")

In [ ]:
# Compare FAISS vs Chroma results for the same query
query_compare = "How do I connect to VPN?"
print(f"Comparing FAISS vs Chroma for: '{query_compare}'\n")

# FAISS
faiss_results = search(query_compare, k=3)
print("FAISS results:")
for r in faiss_results:
    print(f"  {r['rank']}. [{r['title']}]  dist={r['distance']:.4f}")

# Chroma
chroma_results = collection.query(query_texts=[query_compare], n_results=3)
print("\nChroma results:")
for doc, dist, meta in zip(
    chroma_results["documents"][0],
    chroma_results["distances"][0],
    chroma_results["metadatas"][0],
):
    print(f"  [{meta['title']}]  dist={dist:.4f}")

## Complete pipeline as a reusable class

Wrap the entire pipeline in a class for production use.

In [ ]:
class SemanticSearchEngine:
    """Semantic search over a document corpus using FAISS."""
    
    def __init__(self, model_name: str, documents: list[str]):
        self.model = SentenceTransformer(model_name)
        self.documents = documents
        self.titles = [doc.split(":")[0] for doc in documents]
        
        # Embed and index
        self.embeddings = self.model.encode(documents, convert_to_numpy=True).astype("float32")
        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(self.embeddings)
    
    def search(self, query: str, k: int = 3) -> list[dict]:
        q_vec = self.model.encode([query]).astype("float32")
        distances, indices = self.index.search(q_vec, k)
        return [
            {
                "rank": rank + 1,
                "title": self.titles[idx],
                "text": self.documents[idx],
                "distance": float(dist),
            }
            for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]))
        ]
    
    def __repr__(self) -> str:
        return f"SemanticSearchEngine(docs={len(self.documents)}, dim={self.embeddings.shape[1]})"


engine = SemanticSearchEngine(CONFIG["embedding_model"], documents)
print(engine)

In [ ]:
# Use the engine
for q in ["forgot password", "new laptop", "book a room"]:
    results = engine.search(q, k=2)
    print(f"Q: {q}")
    for r in results:
        print(f"  {r['rank']}. [{r['title']}] dist={r['distance']:.4f}")
    print()

In [ ]:
# Simulate a production query loop
user_queries = [
    "Can someone help me with my printer?",
    "What's the policy on remote work?",
    "Is there IT training available?",
    "my screen keeps flickering",
    "how to get access to internal wiki",
]

print("Simulating helpdesk query stream:\n")
for q in user_queries:
    start = time.perf_counter()
    results = engine.search(q, k=1)
    elapsed = (time.perf_counter() - start) * 1000
    r = results[0]
    print(f"  [{elapsed:5.1f}ms] Q: '{q}'")
    print(f"           A: [{r['title']}] dist={r['distance']:.4f}")
    print()

## Success criteria

- Documents embedded and indexed in FAISS (20 articles, 384 dimensions)
- Natural-language queries return top-k semantically similar documents
- Recall@3 measured on ground-truth test cases
- Pipeline matches the diagram: Load → Embed → Index → Query → Search → Display
- Chroma alternative demonstrated with metadata filtering
- Reusable `SemanticSearchEngine` class wraps the full pipeline

Day 3 extends this pipeline: instead of returning documents directly, a **RAG system** retrieves documents and passes them to an LLM to generate an answer.

In [ ]:
# Cleanup Chroma
chroma_client.delete_collection("helpdesk_kb")
chroma_client.delete_collection("helpdesk_kb_categories")
print("Cleanup complete.")